# Chapter 9.2: Load Balancing & Auto-Scaling LLM Services

LLM serving has unique load-balancing challenges: long-lived connections (streaming),
variable request durations, GPU memory constraints, and KV-cache affinity. This notebook
covers practical load-balancing strategies and auto-scaling patterns.

## Learning Objectives
- Configure Nginx as a load balancer for LLM endpoints
- Implement round-robin, least-connections, and KV-cache-aware routing
- Build a custom Python load balancer with FastAPI
- Design auto-scaling policies based on GPU and queue metrics
- Handle session affinity for multi-turn conversations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part9/chapter_9.2_load_balancing.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part9/chapter_9.2_load_balancing.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

In [ ]:
import os
import json
import time
import random
import asyncio
import hashlib
import textwrap
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional
from collections import defaultdict

CONFIG_DIR = Path("./lb_configs")
CONFIG_DIR.mkdir(exist_ok=True)

def write_config(filename, content, subdir=None):
    target_dir = CONFIG_DIR / subdir if subdir else CONFIG_DIR
    target_dir.mkdir(parents=True, exist_ok=True)
    path = target_dir / filename
    path.write_text(textwrap.dedent(content).strip() + "\n")
    print(f"Written: {path}")
    print(path.read_text())
    return path

print("Setup complete.")

## 1. Demo: Nginx Load Balancer for LLM Endpoints

Nginx configuration optimized for LLM serving with streaming support,
long timeouts, and health checks.

In [ ]:
nginx_conf = """\
# Nginx load balancer for vLLM endpoints
# Optimized for LLM serving with streaming (SSE) support

upstream vllm_backends {
    # Strategy: least_conn is better for LLM serving than round-robin
    # because request durations vary wildly (10ms to 60s+)
    least_conn;

    server vllm-1:8000 weight=1 max_fails=3 fail_timeout=30s;
    server vllm-2:8000 weight=1 max_fails=3 fail_timeout=30s;
    server vllm-3:8000 weight=1 max_fails=3 fail_timeout=30s;

    # Keep connections alive to reduce overhead
    keepalive 32;
    keepalive_time 1h;
    keepalive_timeout 60s;
}

# Rate limiting zone
limit_req_zone $binary_remote_addr zone=llm_rate:10m rate=10r/s;

server {
    listen 80;
    server_name llm-api.example.com;

    # Logging
    access_log /var/log/nginx/llm_access.log;
    error_log /var/log/nginx/llm_error.log;

    # Client request limits
    client_max_body_size 10m;       # Large prompts
    client_body_timeout 30s;

    # Proxy settings for LLM serving
    location /v1/ {
        # Rate limiting with burst
        limit_req zone=llm_rate burst=20 nodelay;

        proxy_pass http://vllm_backends;

        # Critical for streaming (SSE)
        proxy_http_version 1.1;
        proxy_set_header Connection "";
        proxy_buffering off;          # Required for streaming!
        proxy_cache off;
        chunked_transfer_encoding on;

        # Long timeouts for LLM generation
        proxy_connect_timeout 10s;
        proxy_read_timeout 300s;      # 5 min for long generations
        proxy_send_timeout 300s;

        # Headers
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Request-ID $request_id;

        # Retry on connection failures (NOT on timeouts)
        proxy_next_upstream error timeout;
        proxy_next_upstream_tries 2;
    }

    # Health check endpoint
    location /health {
        proxy_pass http://vllm_backends/health;
        proxy_connect_timeout 5s;
        proxy_read_timeout 5s;
    }

    # Metrics passthrough
    location /metrics {
        proxy_pass http://vllm_backends/metrics;
        allow 10.0.0.0/8;     # Internal only
        deny all;
    }

    # Stub status for Nginx monitoring
    location /nginx_status {
        stub_status;
        allow 127.0.0.1;
        deny all;
    }
}
"""

write_config("nginx.conf", nginx_conf, subdir="nginx")

In [ ]:
# Docker-compose for Nginx + multiple vLLM backends
lb_compose = """\
version: '3.8'

services:
  nginx:
    image: nginx:1.25
    ports:
      - "80:80"
    volumes:
      - ./nginx.conf:/etc/nginx/conf.d/default.conf:ro
    depends_on:
      vllm-1:
        condition: service_healthy
      vllm-2:
        condition: service_healthy
    restart: unless-stopped

  vllm-1:
    image: vllm/vllm-openai:latest
    runtime: nvidia
    environment:
      - NVIDIA_VISIBLE_DEVICES=0
    command: ["--model", "facebook/opt-1.3b", "--host", "0.0.0.0", "--port", "8000"]
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      start_period: 120s
    deploy:
      resources:
        reservations:
          devices:
            - driver: nvidia
              device_ids: ['0']
              capabilities: [gpu]

  vllm-2:
    image: vllm/vllm-openai:latest
    runtime: nvidia
    environment:
      - NVIDIA_VISIBLE_DEVICES=1
    command: ["--model", "facebook/opt-1.3b", "--host", "0.0.0.0", "--port", "8000"]
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      start_period: 120s
    deploy:
      resources:
        reservations:
          devices:
            - driver: nvidia
              device_ids: ['1']
              capabilities: [gpu]
"""

write_config("docker-compose-lb.yml", lb_compose, subdir="nginx")

## 2. Demo: Load Balancing Strategies Comparison

We simulate different load-balancing strategies and compare their effectiveness
for LLM serving workloads.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import numpy as np

def draw_load_balancing_strategies():
    """Load balancing strategies: Round-robin vs Least-connections vs KV-cache-aware."""
    fig, axes = plt.subplots(1, 3, figsize=(20, 7))

    strategies = [
        ('Round Robin\n(Equal Distribution)', '#4A90D9'),
        ('Least Connections\n(Smart Routing)', '#27AE60'),
        ('KV-Cache Aware\n(Prefix-Based Routing)', '#F39C12'),
    ]

    for ax, (title, color) in zip(axes, strategies):
        ax.set_xlim(0, 10); ax.set_ylim(0, 12); ax.axis('off')
        ax.set_title(title, fontsize=12, fontweight='bold', color=color, pad=10)

        # Load Balancer box
        lb = FancyBboxPatch((2.5, 9.5), 5, 1.5, boxstyle='round,pad=0.15',
                            facecolor=color, edgecolor='white', linewidth=2, alpha=0.85)
        ax.add_patch(lb)
        ax.text(5, 10.25, 'Load Balancer', ha='center', va='center',
                fontsize=10, fontweight='bold', color='white')

        # Client requests at top
        for i in range(4):
            ax.annotate('', xy=(3 + i * 1.2, 9.5), xytext=(3 + i * 1.2, 11.5),
                        arrowprops=dict(arrowstyle='->', color='#666', lw=1.5))
            ax.text(3 + i * 1.2, 11.6, f'R{i+1}', ha='center', fontsize=8, color='#666')

    # --- Round Robin ---
    ax = axes[0]
    gpu_loads = [2, 2, 2]  # Equal
    for i, load in enumerate(gpu_loads):
        y_pos = 2.0 + i * 2.5
        gpu = FancyBboxPatch((2.5, y_pos), 5, 1.8, boxstyle='round,pad=0.1',
                             facecolor='#4A90D9', edgecolor='white', linewidth=1.5, alpha=0.7)
        ax.add_patch(gpu)
        fill_w = 5 * (load / 4)
        fill = FancyBboxPatch((2.5, y_pos), fill_w, 1.8, boxstyle='round,pad=0.1',
                              facecolor='#4A90D9', edgecolor='none', alpha=0.9)
        ax.add_patch(fill)
        ax.text(5, y_pos + 0.9, f'GPU-{i}  ({load} req)', ha='center', va='center',
                fontsize=9, fontweight='bold', color='white')
        ax.annotate('', xy=(5, y_pos + 1.8), xytext=(5, 9.5),
                    arrowprops=dict(arrowstyle='->', color='#4A90D9', lw=1.5, alpha=0.5))

    # --- Least Connections ---
    ax = axes[1]
    gpu_loads = [1, 3, 0]  # Unequal incoming -> route to least loaded
    for i, load in enumerate(gpu_loads):
        y_pos = 2.0 + i * 2.5
        gpu = FancyBboxPatch((2.5, y_pos), 5, 1.8, boxstyle='round,pad=0.1',
                             facecolor='#27AE60', edgecolor='white', linewidth=1.5, alpha=0.7)
        ax.add_patch(gpu)
        fill_w = 5 * (load / 4)
        fill = FancyBboxPatch((2.5, y_pos), max(0.2, fill_w), 1.8, boxstyle='round,pad=0.1',
                              facecolor='#27AE60', edgecolor='none', alpha=0.9)
        ax.add_patch(fill)
        marker = ' <-- next' if load == 0 else ''
        ax.text(5, y_pos + 0.9, f'GPU-{i}  ({load} active){marker}', ha='center', va='center',
                fontsize=9, fontweight='bold', color='white')
        lw = 2.5 if load == 0 else 1.0
        ax.annotate('', xy=(5, y_pos + 1.8), xytext=(5, 9.5),
                    arrowprops=dict(arrowstyle='->', color='#27AE60', lw=lw, alpha=0.7))

    # --- KV-Cache Aware ---
    ax = axes[2]
    gpu_info = [('prefix-A cached', 1, 0.3), ('prefix-B cached', 2, 0.7), ('empty', 0, 0.05)]
    for i, (cache_info, load, kv) in enumerate(gpu_info):
        y_pos = 2.0 + i * 2.5
        gpu = FancyBboxPatch((2.5, y_pos), 5, 1.8, boxstyle='round,pad=0.1',
                             facecolor='#F39C12', edgecolor='white', linewidth=1.5, alpha=0.7)
        ax.add_patch(gpu)
        fill_w = 5 * kv
        fill = FancyBboxPatch((2.5, y_pos), max(0.2, fill_w), 1.8, boxstyle='round,pad=0.1',
                              facecolor='#E74C3C', edgecolor='none', alpha=0.4)
        ax.add_patch(fill)
        ax.text(5, y_pos + 1.1, f'GPU-{i}  (KV: {kv:.0%})', ha='center', va='center',
                fontsize=9, fontweight='bold', color='white')
        ax.text(5, y_pos + 0.4, cache_info, ha='center', va='center',
                fontsize=7, color='white', style='italic')

    plt.tight_layout()
    plt.savefig('load_balancing_strategies.png', dpi=150, bbox_inches='tight')
    plt.show()

draw_load_balancing_strategies()

In [ ]:
@dataclass
class Backend:
    """Simulated LLM backend server."""
    name: str
    active_requests: int = 0
    total_requests: int = 0
    kv_cache_usage: float = 0.0  # 0.0 to 1.0
    total_latency: float = 0.0
    gpu_utilization: float = 0.0
    
    def process_request(self, input_tokens: int, output_tokens: int) -> float:
        """Simulate processing a request. Returns latency."""
        self.active_requests += 1
        self.total_requests += 1
        
        # Simulate: latency increases with concurrent requests and KV cache pressure
        base_latency = (input_tokens * 0.001 + output_tokens * 0.01)
        congestion_factor = 1.0 + (self.active_requests * 0.3)
        cache_factor = 1.0 + max(0, (self.kv_cache_usage - 0.7)) * 5
        
        latency = base_latency * congestion_factor * cache_factor
        
        # Update KV cache usage
        self.kv_cache_usage = min(1.0, self.kv_cache_usage + (input_tokens + output_tokens) * 0.0001)
        self.gpu_utilization = min(1.0, self.active_requests * 0.15 + 0.1)
        self.total_latency += latency
        
        return latency
    
    def finish_request(self, tokens_freed: int = 500):
        self.active_requests = max(0, self.active_requests - 1)
        self.kv_cache_usage = max(0, self.kv_cache_usage - tokens_freed * 0.0001)


class LoadBalancer:
    """Base load balancer."""
    def __init__(self, backends: list[Backend]):
        self.backends = backends
    
    def select_backend(self, request: dict) -> Backend:
        raise NotImplementedError


class RoundRobinLB(LoadBalancer):
    def __init__(self, backends):
        super().__init__(backends)
        self._index = 0
    
    def select_backend(self, request: dict) -> Backend:
        backend = self.backends[self._index % len(self.backends)]
        self._index += 1
        return backend


class LeastConnectionsLB(LoadBalancer):
    def select_backend(self, request: dict) -> Backend:
        return min(self.backends, key=lambda b: b.active_requests)


class KVCacheAwareLB(LoadBalancer):
    """Route to backend with lowest KV cache usage, weighted by active requests."""
    def select_backend(self, request: dict) -> Backend:
        def score(b: Backend) -> float:
            # Lower score = better candidate
            return b.kv_cache_usage * 0.6 + (b.active_requests / 10) * 0.4
        return min(self.backends, key=score)


print("Load balancer classes defined. See next cell for simulation.")

In [ ]:
def simulate_workload(lb: LoadBalancer, num_requests: int = 200, seed: int = 42) -> dict:
    """Simulate a bursty LLM workload and collect metrics."""
    rng = random.Random(seed)
    latencies = []
    
    for i in range(num_requests):
        # Simulate variable request sizes
        input_tokens = rng.randint(50, 2000)
        output_tokens = rng.randint(20, 500)
        
        request = {"input_tokens": input_tokens, "output_tokens": output_tokens}
        backend = lb.select_backend(request)
        latency = backend.process_request(input_tokens, output_tokens)
        latencies.append(latency)
        
        # Some requests finish (simulate concurrent processing)
        if i > 0 and rng.random() < 0.7:
            finish_backend = rng.choice(lb.backends)
            if finish_backend.active_requests > 0:
                finish_backend.finish_request()
    
    # Compute statistics
    latencies_sorted = sorted(latencies)
    return {
        "avg_latency": sum(latencies) / len(latencies),
        "p50_latency": latencies_sorted[len(latencies) // 2],
        "p99_latency": latencies_sorted[int(len(latencies) * 0.99)],
        "max_latency": max(latencies),
        "distribution": {
            b.name: {"requests": b.total_requests, "kv_cache": round(b.kv_cache_usage, 3)}
            for b in lb.backends
        }
    }


# Compare strategies
strategies = {
    "Round Robin": RoundRobinLB,
    "Least Connections": LeastConnectionsLB,
    "KV-Cache Aware": KVCacheAwareLB,
}

print(f"{'Strategy':<20} {'Avg Latency':>12} {'P50':>10} {'P99':>10} {'Max':>10}")
print("-" * 65)

results = {}
for name, LBClass in strategies.items():
    backends = [Backend(f"gpu-{i}") for i in range(3)]
    lb = LBClass(backends)
    result = simulate_workload(lb, num_requests=500)
    results[name] = result
    print(f"{name:<20} {result['avg_latency']:>10.2f}ms {result['p50_latency']:>10.2f} "
          f"{result['p99_latency']:>10.2f} {result['max_latency']:>10.2f}")

print("\nRequest Distribution (per backend):")
for name, result in results.items():
    dist = result["distribution"]
    counts = [f"{k}: {v['requests']}" for k, v in dist.items()]
    print(f"  {name:<20} {', '.join(counts)}")

## 3. Demo: Custom Load Balancer with FastAPI

A production-ready load balancer that handles health checks, session affinity,
and streaming proxying.

In [ ]:
# Custom FastAPI-based LLM load balancer

fastapi_lb_code = '''\
"""Custom LLM Load Balancer with KV-cache awareness and session affinity."""

import asyncio
import hashlib
import time
import json
from dataclasses import dataclass, field
from typing import Optional
from contextlib import asynccontextmanager

import httpx
from fastapi import FastAPI, Request, Response
from fastapi.responses import StreamingResponse


@dataclass
class BackendState:
    """Track the state of a backend for intelligent routing."""
    url: str
    healthy: bool = True
    active_requests: int = 0
    kv_cache_usage: float = 0.0
    gpu_utilization: float = 0.0
    last_health_check: float = 0.0
    total_requests: int = 0
    total_errors: int = 0
    avg_latency_ms: float = 0.0

    def routing_score(self) -> float:
        """Lower score = better routing candidate."""
        if not self.healthy:
            return float("inf")
        return (
            self.kv_cache_usage * 0.4
            + (self.active_requests / max(1, 10)) * 0.3
            + self.gpu_utilization * 0.2
            + (self.avg_latency_ms / max(1, 5000)) * 0.1
        )


class LLMLoadBalancer:
    """Intelligent load balancer for LLM serving."""

    def __init__(self, backend_urls: list[str]):
        self.backends = {url: BackendState(url=url) for url in backend_urls}
        self.session_map: dict[str, str] = {}  # session_id -> backend_url
        self.client = httpx.AsyncClient(timeout=300.0)
        self._health_task: Optional[asyncio.Task] = None

    async def start_health_checks(self, interval: float = 10.0):
        """Periodically check backend health and collect metrics."""
        async def _loop():
            while True:
                for url, state in self.backends.items():
                    try:
                        resp = await self.client.get(f"{url}/health", timeout=5.0)
                        state.healthy = resp.status_code == 200
                    except Exception:
                        state.healthy = False

                    # Fetch metrics if healthy
                    if state.healthy:
                        try:
                            resp = await self.client.get(f"{url}/metrics", timeout=5.0)
                            text = resp.text
                            # Parse key metrics from Prometheus format
                            for line in text.split("\\n"):
                                if "gpu_cache_usage_perc" in line and not line.startswith("#"):
                                    state.kv_cache_usage = float(line.split()[-1])
                                elif "num_requests_running" in line and not line.startswith("#"):
                                    state.active_requests = int(float(line.split()[-1]))
                        except Exception:
                            pass

                    state.last_health_check = time.time()
                await asyncio.sleep(interval)

        self._health_task = asyncio.create_task(_loop())

    def select_backend(self, session_id: Optional[str] = None) -> BackendState:
        """Select the best backend for a request."""
        # Session affinity: return same backend for same session
        if session_id and session_id in self.session_map:
            url = self.session_map[session_id]
            if self.backends[url].healthy:
                return self.backends[url]

        # KV-cache-aware routing: pick lowest score
        healthy = [b for b in self.backends.values() if b.healthy]
        if not healthy:
            raise RuntimeError("No healthy backends available")

        best = min(healthy, key=lambda b: b.routing_score())

        if session_id:
            self.session_map[session_id] = best.url

        return best

    async def proxy_request(self, request: Request) -> Response:
        """Proxy a request to the selected backend."""
        # Extract session ID for affinity
        session_id = request.headers.get("X-Session-ID")
        backend = self.select_backend(session_id)
        backend.active_requests += 1
        backend.total_requests += 1

        start = time.time()
        try:
            body = await request.body()
            body_json = json.loads(body) if body else {}
            is_streaming = body_json.get("stream", False)

            url = f"{backend.url}{request.url.path}"
            headers = dict(request.headers)
            headers.pop("host", None)

            if is_streaming:
                # Streaming proxy
                req = self.client.build_request(
                    method=request.method, url=url,
                    content=body, headers=headers
                )
                resp = await self.client.send(req, stream=True)

                async def stream_generator():
                    try:
                        async for chunk in resp.aiter_bytes():
                            yield chunk
                    finally:
                        await resp.aclose()
                        backend.active_requests -= 1
                        elapsed = (time.time() - start) * 1000
                        backend.avg_latency_ms = (
                            backend.avg_latency_ms * 0.9 + elapsed * 0.1
                        )

                return StreamingResponse(
                    stream_generator(),
                    status_code=resp.status_code,
                    media_type="text/event-stream"
                )
            else:
                # Non-streaming proxy
                resp = await self.client.request(
                    method=request.method, url=url,
                    content=body, headers=headers
                )
                backend.active_requests -= 1
                elapsed = (time.time() - start) * 1000
                backend.avg_latency_ms = (
                    backend.avg_latency_ms * 0.9 + elapsed * 0.1
                )
                return Response(
                    content=resp.content,
                    status_code=resp.status_code,
                    headers=dict(resp.headers)
                )
        except Exception as e:
            backend.active_requests -= 1
            backend.total_errors += 1
            return Response(
                content=json.dumps({"error": str(e)}),
                status_code=502
            )


# --- FastAPI App ---

BACKEND_URLS = [
    "http://vllm-1:8000",
    "http://vllm-2:8000",
    "http://vllm-3:8000",
]

lb = LLMLoadBalancer(BACKEND_URLS)

@asynccontextmanager
async def lifespan(app: FastAPI):
    await lb.start_health_checks()
    yield
    if lb._health_task:
        lb._health_task.cancel()

app = FastAPI(title="LLM Load Balancer", lifespan=lifespan)

@app.api_route("/v1/{path:path}", methods=["GET", "POST"])
async def proxy(request: Request):
    return await lb.proxy_request(request)

@app.get("/lb/status")
async def lb_status():
    return {
        "backends": [
            {
                "url": b.url,
                "healthy": b.healthy,
                "active_requests": b.active_requests,
                "total_requests": b.total_requests,
                "kv_cache_usage": round(b.kv_cache_usage, 3),
                "routing_score": round(b.routing_score(), 3),
                "avg_latency_ms": round(b.avg_latency_ms, 1),
            }
            for b in lb.backends.values()
        ]
    }

@app.get("/health")
async def health():
    healthy_count = sum(1 for b in lb.backends.values() if b.healthy)
    return {"status": "ok" if healthy_count > 0 else "unhealthy",
            "healthy_backends": healthy_count}
'''

write_config("llm_load_balancer.py", fastapi_lb_code)
print("\nRun with: uvicorn llm_load_balancer:app --host 0.0.0.0 --port 8080")

## 4. Auto-Scaling Based on GPU Utilization and Queue Depth

In [ ]:
@dataclass
class ScalingMetrics:
    """Metrics used for scaling decisions."""
    gpu_utilization: float        # 0-1
    kv_cache_usage: float         # 0-1
    queue_depth: int              # pending requests
    avg_latency_ms: float         # recent average
    requests_per_second: float    # recent throughput


class AutoScaler:
    """Auto-scaler for LLM serving replicas."""
    
    def __init__(
        self,
        min_replicas: int = 1,
        max_replicas: int = 10,
        scale_up_threshold: float = 0.8,   # GPU util threshold
        scale_down_threshold: float = 0.3,
        queue_depth_per_replica: int = 20,  # Target queue per replica
        cooldown_seconds: float = 300,
    ):
        self.min_replicas = min_replicas
        self.max_replicas = max_replicas
        self.scale_up_threshold = scale_up_threshold
        self.scale_down_threshold = scale_down_threshold
        self.queue_depth_per_replica = queue_depth_per_replica
        self.cooldown_seconds = cooldown_seconds
        
        self.current_replicas = min_replicas
        self.last_scale_time = 0.0
        self.decisions_log: list[dict] = []
    
    def compute_desired_replicas(self, metrics: ScalingMetrics) -> int:
        """Compute desired replica count based on metrics."""
        candidates = []
        
        # Strategy 1: GPU utilization based
        if metrics.gpu_utilization > 0:
            gpu_desired = int(
                self.current_replicas * metrics.gpu_utilization / self.scale_up_threshold
            )
            candidates.append(("gpu_util", gpu_desired))
        
        # Strategy 2: Queue depth based
        if metrics.queue_depth > 0:
            queue_desired = max(1, metrics.queue_depth // self.queue_depth_per_replica + 1)
            candidates.append(("queue_depth", queue_desired))
        
        # Strategy 3: KV cache pressure
        if metrics.kv_cache_usage > 0.9:
            kv_desired = self.current_replicas + 1
            candidates.append(("kv_cache", kv_desired))
        
        # Take the maximum of all strategies (scale for the bottleneck)
        if candidates:
            desired = max(c[1] for c in candidates)
        else:
            desired = self.current_replicas
        
        # Clamp to bounds
        desired = max(self.min_replicas, min(self.max_replicas, desired))
        return desired
    
    def should_scale(self, metrics: ScalingMetrics) -> tuple[bool, int, str]:
        """Decide whether to scale and by how much."""
        now = time.time()
        
        # Cooldown check
        if now - self.last_scale_time < self.cooldown_seconds:
            return False, self.current_replicas, "cooldown"
        
        desired = self.compute_desired_replicas(metrics)
        
        if desired > self.current_replicas:
            reason = f"scale up {self.current_replicas} -> {desired}"
            return True, desired, reason
        elif desired < self.current_replicas and metrics.gpu_utilization < self.scale_down_threshold:
            reason = f"scale down {self.current_replicas} -> {desired}"
            return True, desired, reason
        
        return False, self.current_replicas, "no change needed"
    
    def apply_scaling(self, metrics: ScalingMetrics) -> dict:
        """Apply scaling decision."""
        should, desired, reason = self.should_scale(metrics)
        decision = {
            "timestamp": time.time(),
            "current": self.current_replicas,
            "desired": desired,
            "action": reason,
            "applied": should,
            "metrics": {
                "gpu_util": metrics.gpu_utilization,
                "kv_cache": metrics.kv_cache_usage,
                "queue": metrics.queue_depth,
            }
        }
        
        if should:
            self.current_replicas = desired
            self.last_scale_time = time.time()
        
        self.decisions_log.append(decision)
        return decision


# Simulate auto-scaling over time
scaler = AutoScaler(min_replicas=1, max_replicas=8, cooldown_seconds=0)  # 0 for demo

# Simulate a workload ramp-up then cool-down
scenarios = [
    ("Low load",     ScalingMetrics(0.2, 0.1, 5,  100, 10)),
    ("Medium load",  ScalingMetrics(0.5, 0.4, 25, 200, 30)),
    ("High load",    ScalingMetrics(0.85, 0.7, 80, 500, 50)),
    ("Peak load",    ScalingMetrics(0.95, 0.92, 150, 1200, 60)),
    ("Cooling down", ScalingMetrics(0.4, 0.3, 10, 150, 20)),
    ("Low again",    ScalingMetrics(0.15, 0.1, 2, 80, 5)),
]

print(f"{'Scenario':<15} {'Replicas':>8} {'Action':<35} {'GPU':>5} {'Queue':>6}")
print("-" * 75)
for name, metrics in scenarios:
    decision = scaler.apply_scaling(metrics)
    print(f"{name:<15} {decision['desired']:>8} {decision['action']:<35} "
          f"{metrics.gpu_utilization:>5.0%} {metrics.queue_depth:>6}")

## 5. Session Affinity for Multi-Turn Conversations

In [ ]:
import hashlib
from collections import defaultdict

class SessionAffinityRouter:
    """Route multi-turn conversations to the same backend for KV cache reuse."""
    
    def __init__(self, backends: list[str], max_sessions_per_backend: int = 100):
        self.backends = backends
        self.max_sessions = max_sessions_per_backend
        self.session_map: dict[str, str] = {}        # session -> backend
        self.backend_sessions: dict[str, set] = defaultdict(set)  # backend -> sessions
        self.session_last_seen: dict[str, float] = {}
    
    def _hash_route(self, session_id: str) -> str:
        """Consistent hash to assign sessions to backends."""
        h = int(hashlib.md5(session_id.encode()).hexdigest(), 16)
        return self.backends[h % len(self.backends)]
    
    def route(self, session_id: str) -> str:
        """Get the backend for a session."""
        now = time.time()
        
        # Check existing mapping
        if session_id in self.session_map:
            backend = self.session_map[session_id]
            self.session_last_seen[session_id] = now
            return backend
        
        # New session: use consistent hashing
        backend = self._hash_route(session_id)
        
        # Check capacity
        if len(self.backend_sessions[backend]) >= self.max_sessions:
            # Overflow: find least loaded backend
            backend = min(self.backends, key=lambda b: len(self.backend_sessions[b]))
        
        self.session_map[session_id] = backend
        self.backend_sessions[backend].add(session_id)
        self.session_last_seen[session_id] = now
        return backend
    
    def cleanup_stale(self, max_age_seconds: float = 3600):
        """Remove sessions not seen for max_age_seconds."""
        now = time.time()
        stale = [
            sid for sid, ts in self.session_last_seen.items()
            if now - ts > max_age_seconds
        ]
        for sid in stale:
            backend = self.session_map.pop(sid, None)
            if backend:
                self.backend_sessions[backend].discard(sid)
            self.session_last_seen.pop(sid, None)
        return len(stale)
    
    def status(self) -> dict:
        return {
            b: len(self.backend_sessions[b]) for b in self.backends
        }


# Demo: simulate multi-turn conversations
router = SessionAffinityRouter(["gpu-0", "gpu-1", "gpu-2"])

# Simulate 20 conversations, each with multiple turns
print("Session affinity demo:")
print(f"{'Session':<12} {'Turn':>5} {'Backend':<10} {'Affinity'}")
print("-" * 45)

for conv_id in range(6):
    session = f"conv-{conv_id}"
    for turn in range(3):
        backend = router.route(session)
        affinity = "(new)" if turn == 0 else "(cached)"
        print(f"{session:<12} {turn:>5} {backend:<10} {affinity}")

print(f"\nSession distribution: {router.status()}")

---
## Hands-on Assignments

### Assignment 1: Implement a KV-Cache-Aware Load Balancer

Build a load balancer that queries each backend's `/metrics` endpoint to read
KV cache utilization and routes to the backend with the most available cache space.

In [ ]:
# ASSIGNMENT 1: KV-Cache-Aware Load Balancer

@dataclass
class KVBackend:
    url: str
    kv_cache_usage: float = 0.0     # 0.0 to 1.0
    active_requests: int = 0
    max_kv_blocks: int = 10000
    used_kv_blocks: int = 0
    healthy: bool = True


class KVCacheAwareRouter:
    """Route requests to the backend with the most available KV cache space.
    
    Your implementation should:
    1. Select the backend with lowest KV cache usage among healthy backends
    2. If two backends have similar KV cache usage (<5% difference),
       prefer the one with fewer active requests
    3. If a request includes a 'prefix_hash', route to a backend that might
       have that prefix cached (consistent hashing)
    4. Never route to unhealthy backends
    """
    
    def __init__(self, backends: list[KVBackend]):
        self.backends = backends
        self.prefix_map: dict[str, str] = {}  # prefix_hash -> backend_url
    
    def route(self, prefix_hash: str = None) -> KVBackend:
        """Select the best backend.
        
        TODO: Implement the routing logic described above.
        """
        # YOUR CODE HERE
        pass
    
    def update_metrics(self, backend_url: str, kv_usage: float, active: int):
        """Update backend metrics (simulating a metrics scrape)."""
        for b in self.backends:
            if b.url == backend_url:
                b.kv_cache_usage = kv_usage
                b.active_requests = active
                break


# Test your implementation
backends = [
    KVBackend("http://gpu-0:8000", kv_cache_usage=0.3, active_requests=5),
    KVBackend("http://gpu-1:8000", kv_cache_usage=0.7, active_requests=2),
    KVBackend("http://gpu-2:8000", kv_cache_usage=0.31, active_requests=8),  # Similar to gpu-0
]

router = KVCacheAwareRouter(backends)

# Test 1: Should pick gpu-0 (lowest KV cache, and fewer requests than gpu-2)
result = router.route()
print(f"Test 1 - Lowest KV cache: {result.url if result else 'None'}")
print(f"  Expected: http://gpu-0:8000")

# Test 2: With prefix hash that maps to gpu-1
result = router.route(prefix_hash="system_prompt_v1")
print(f"\nTest 2 - With prefix: {result.url if result else 'None'}")

# Test 3: Make gpu-0 unhealthy
backends[0].healthy = False
result = router.route()
print(f"\nTest 3 - With gpu-0 down: {result.url if result else 'None'}")
print(f"  Expected: http://gpu-2:8000 (next lowest KV cache)")

In [ ]:
# ASSIGNMENT 1 - SOLUTION

class KVCacheAwareRouterSolution:
    def __init__(self, backends: list[KVBackend]):
        self.backends = backends
        self.prefix_map: dict[str, str] = {}
    
    def route(self, prefix_hash: str = None) -> KVBackend:
        healthy = [b for b in self.backends if b.healthy]
        if not healthy:
            raise RuntimeError("No healthy backends")
        
        # Check prefix affinity first
        if prefix_hash and prefix_hash in self.prefix_map:
            cached_url = self.prefix_map[prefix_hash]
            for b in healthy:
                if b.url == cached_url and b.kv_cache_usage < 0.9:
                    return b
        
        # Sort by KV cache usage
        sorted_backends = sorted(healthy, key=lambda b: b.kv_cache_usage)
        
        # If top candidates are within 5%, pick by active requests
        best = sorted_backends[0]
        similar = [b for b in sorted_backends 
                   if abs(b.kv_cache_usage - best.kv_cache_usage) < 0.05]
        
        if len(similar) > 1:
            best = min(similar, key=lambda b: b.active_requests)
        
        # Store prefix mapping
        if prefix_hash:
            self.prefix_map[prefix_hash] = best.url
        
        return best

# Re-test with solution
backends = [
    KVBackend("http://gpu-0:8000", kv_cache_usage=0.3, active_requests=5),
    KVBackend("http://gpu-1:8000", kv_cache_usage=0.7, active_requests=2),
    KVBackend("http://gpu-2:8000", kv_cache_usage=0.31, active_requests=8),
]

router = KVCacheAwareRouterSolution(backends)
result = router.route()
print(f"Test 1: {result.url} (expected gpu-0, fewer requests among similar KV usage)")

result = router.route(prefix_hash="system_v1")
print(f"Test 2: {result.url} (with prefix, stored for future affinity)")

result = router.route(prefix_hash="system_v1")  # Should reuse
print(f"Test 3: {result.url} (prefix affinity reuse)")

backends[0].healthy = False
result = router.route()
print(f"Test 4: {result.url} (gpu-0 down, should pick gpu-2)")

### Assignment 2: Build an Auto-Scaler That Monitors Queue Depth

In [ ]:
# ASSIGNMENT 2: Queue-depth-based auto-scaler with smoothing

class QueueDepthAutoScaler:
    """Auto-scale replicas based on request queue depth.
    
    Requirements:
    1. Use exponential moving average (EMA) of queue depth to avoid noise
    2. Scale up when EMA exceeds target_queue_per_replica * current_replicas
    3. Scale down when EMA is below target_queue_per_replica * (current_replicas - 1)
    4. Enforce cooldown between scaling events
    5. Never scale below min_replicas or above max_replicas
    6. Log all scaling decisions with reasons
    """
    
    def __init__(
        self,
        min_replicas: int = 1,
        max_replicas: int = 10,
        target_queue_per_replica: int = 10,
        ema_alpha: float = 0.3,
        scale_up_cooldown: float = 60,
        scale_down_cooldown: float = 300,
    ):
        self.min_replicas = min_replicas
        self.max_replicas = max_replicas
        self.target_queue = target_queue_per_replica
        self.ema_alpha = ema_alpha
        self.scale_up_cooldown = scale_up_cooldown
        self.scale_down_cooldown = scale_down_cooldown
        
        self.current_replicas = min_replicas
        self.ema_queue_depth = 0.0
        self.last_scale_up_time = 0.0
        self.last_scale_down_time = 0.0
        self.log: list[dict] = []
    
    def observe(self, queue_depth: int, timestamp: float = None) -> dict:
        """Observe current queue depth and decide on scaling.
        
        TODO: Implement the scaling logic.
        
        Returns:
            dict with keys: action, replicas, ema_queue, reason
        """
        ts = timestamp or time.time()
        
        # TODO: Update EMA
        # TODO: Compute desired replicas
        # TODO: Check cooldown
        # TODO: Apply scaling
        # TODO: Log decision
        
        return {"action": "none", "replicas": self.current_replicas}


# Test with simulated workload pattern
scaler = QueueDepthAutoScaler(
    min_replicas=1, max_replicas=8,
    target_queue_per_replica=10,
    scale_up_cooldown=0, scale_down_cooldown=0  # Disable for testing
)

# Simulate: ramp up, peak, ramp down
queue_pattern = [5, 8, 15, 25, 40, 60, 80, 100, 80, 50, 30, 15, 8, 3, 2]

print(f"{'Time':>5} {'Queue':>6} {'EMA':>8} {'Replicas':>9} {'Action':<20}")
print("-" * 55)
for t, q in enumerate(queue_pattern):
    result = scaler.observe(q, timestamp=t * 60)
    print(f"{t*60:>5}s {q:>6} {result.get('ema_queue', 0):>8.1f} "
          f"{result['replicas']:>9} {result['action']:<20}")

In [ ]:
# ASSIGNMENT 2 - SOLUTION

class QueueDepthAutoScalerSolution:
    def __init__(self, min_replicas=1, max_replicas=10, target_queue_per_replica=10,
                 ema_alpha=0.3, scale_up_cooldown=60, scale_down_cooldown=300):
        self.min_replicas = min_replicas
        self.max_replicas = max_replicas
        self.target_queue = target_queue_per_replica
        self.ema_alpha = ema_alpha
        self.scale_up_cooldown = scale_up_cooldown
        self.scale_down_cooldown = scale_down_cooldown
        self.current_replicas = min_replicas
        self.ema_queue_depth = 0.0
        self.last_scale_up_time = -9999
        self.last_scale_down_time = -9999
    
    def observe(self, queue_depth: int, timestamp: float = None) -> dict:
        ts = timestamp or time.time()
        
        # Update EMA
        self.ema_queue_depth = (self.ema_alpha * queue_depth + 
                                (1 - self.ema_alpha) * self.ema_queue_depth)
        
        # Compute desired replicas
        desired = max(1, int(self.ema_queue_depth / self.target_queue) + 1)
        desired = max(self.min_replicas, min(self.max_replicas, desired))
        
        action = "none"
        reason = "stable"
        
        if desired > self.current_replicas:
            if ts - self.last_scale_up_time >= self.scale_up_cooldown:
                action = "scale_up"
                reason = f"{self.current_replicas}->{desired} (ema={self.ema_queue_depth:.1f})"
                self.current_replicas = desired
                self.last_scale_up_time = ts
            else:
                reason = "cooldown"
        elif desired < self.current_replicas:
            if ts - self.last_scale_down_time >= self.scale_down_cooldown:
                action = "scale_down"
                reason = f"{self.current_replicas}->{desired} (ema={self.ema_queue_depth:.1f})"
                self.current_replicas = desired
                self.last_scale_down_time = ts
            else:
                reason = "cooldown"
        
        return {
            "action": action,
            "replicas": self.current_replicas,
            "ema_queue": self.ema_queue_depth,
            "reason": reason
        }

scaler = QueueDepthAutoScalerSolution(
    min_replicas=1, max_replicas=8,
    target_queue_per_replica=10,
    scale_up_cooldown=0, scale_down_cooldown=0
)

queue_pattern = [5, 8, 15, 25, 40, 60, 80, 100, 80, 50, 30, 15, 8, 3, 2]

print(f"{'Time':>5} {'Queue':>6} {'EMA':>8} {'Replicas':>9} {'Action':<12} {'Reason'}")
print("-" * 70)
for t, q in enumerate(queue_pattern):
    result = scaler.observe(q, timestamp=t * 60)
    print(f"{t*60:>5}s {q:>6} {result['ema_queue']:>8.1f} "
          f"{result['replicas']:>9} {result['action']:<12} {result['reason']}")

### Assignment 3: Compare Load Balancing Strategies Under Load

In [ ]:
# ASSIGNMENT 3: Run a comprehensive comparison of load balancing strategies

def run_comparison(
    num_requests: int = 1000,
    num_backends: int = 4,
    workload: str = "uniform",  # "uniform", "bursty", "skewed"
) -> dict:
    """Compare load balancing strategies under different workload patterns.
    
    TODO: Implement three workload patterns:
    - uniform: all requests have similar input/output token counts
    - bursty: requests arrive in bursts with periods of quiet
    - skewed: 80% short requests, 20% very long requests
    
    For each pattern, run all three LB strategies and collect:
    - Average latency
    - P50, P95, P99 latency
    - Max KV cache usage across backends
    - Standard deviation of requests per backend (fairness)
    
    Returns:
        dict mapping strategy name to metrics dict
    """
    import statistics
    rng = random.Random(42)
    
    def generate_requests(pattern, n):
        requests = []
        for i in range(n):
            if pattern == "uniform":
                inp = rng.randint(200, 400)
                out = rng.randint(100, 200)
            elif pattern == "bursty":
                # Bursts of 20 requests, then 5 quiet ones
                if i % 25 < 20:
                    inp = rng.randint(100, 1500)
                    out = rng.randint(50, 500)
                else:
                    inp = rng.randint(50, 100)
                    out = rng.randint(10, 30)
            elif pattern == "skewed":
                if rng.random() < 0.8:
                    inp = rng.randint(50, 200)
                    out = rng.randint(20, 50)
                else:
                    inp = rng.randint(1000, 4000)
                    out = rng.randint(200, 1000)
            requests.append({"input_tokens": inp, "output_tokens": out})
        return requests
    
    requests_list = generate_requests(workload, num_requests)
    
    # TODO: Run each strategy and collect metrics
    # Strategies: RoundRobinLB, LeastConnectionsLB, KVCacheAwareLB
    
    results = {}
    for name, LBClass in strategies.items():
        backends = [Backend(f"gpu-{i}") for i in range(num_backends)]
        lb = LBClass(backends)
        latencies = []
        
        for req in requests_list:
            backend = lb.select_backend(req)
            lat = backend.process_request(req["input_tokens"], req["output_tokens"])
            latencies.append(lat)
            if rng.random() < 0.7:
                b = rng.choice(backends)
                if b.active_requests > 0:
                    b.finish_request()
        
        latencies.sort()
        req_counts = [b.total_requests for b in backends]
        
        results[name] = {
            "avg": statistics.mean(latencies),
            "p50": latencies[len(latencies)//2],
            "p95": latencies[int(len(latencies)*0.95)],
            "p99": latencies[int(len(latencies)*0.99)],
            "max_kv": max(b.kv_cache_usage for b in backends),
            "fairness_std": statistics.stdev(req_counts) if len(req_counts) > 1 else 0,
            "distribution": req_counts,
        }
    
    return results


# Run comparisons for all workload types
for workload in ["uniform", "bursty", "skewed"]:
    print(f"\n{'='*70}")
    print(f"Workload: {workload.upper()}")
    print(f"{'='*70}")
    results = run_comparison(num_requests=500, num_backends=4, workload=workload)
    
    print(f"{'Strategy':<20} {'Avg':>8} {'P50':>8} {'P95':>8} {'P99':>8} "
          f"{'MaxKV':>7} {'Fair':>7} {'Distribution'}")
    print("-" * 95)
    for name, m in results.items():
        dist = str(m['distribution'])
        print(f"{name:<20} {m['avg']:>8.1f} {m['p50']:>8.1f} {m['p95']:>8.1f} "
              f"{m['p99']:>8.1f} {m['max_kv']:>6.1%} {m['fairness_std']:>7.1f} {dist}")

## Summary

Key takeaways:

1. **Least-connections outperforms round-robin** for LLM serving due to variable request durations
2. **KV-cache-aware routing** further improves by avoiding backends with high cache pressure
3. **Session affinity** via consistent hashing enables KV cache reuse across multi-turn conversations
4. **Auto-scaling** should use EMA smoothing and separate cooldowns for scale-up vs scale-down
5. **Nginx proxy_buffering must be OFF** for SSE/streaming to work correctly
6. **Queue depth** is often a better scaling signal than CPU utilization for LLM workloads